# **<font color="red">Audio Generation</font>**
- The Gemini API can transform text input into single speaker or multi-speaker audio using Gemini text-to-speech (TTS) generation capabilities. Text-to-speech (TTS) generation is controllable, meaning you can use natural language to structure interactions and guide the _style_, _accent_, _pace_ and _tone_ of the audio.
- **Before you begin:** Ensure you use a Gemini 2.5 model variant with Gemini text-to-speech (TTS) capabilities.

## **<font color="blue">Single-Speaker TTS</font>**
- To convert text to single-speaker audio, set the response modality to "audio", and pass a `SpeechConfig` object with `VoiceConfig` set. You will need to choose a voice name from the prebuilt output voices.

In [ ]:
import os
import asyncio
from google.adk.sessions import InMemorySessionService
from google.adk.artifacts import InMemoryArtifactService
from google.adk.runners import Runner
from google.adk.agents import LlmAgent
import google.genai.types as types
from config import config

# -----------------------------
# Setup API Key and model
# -----------------------------
APP_NAME = "chatbot_demo"
USER_ID = "user_001"
SESSION_ID = "session_001"
MODEL = "gemini-2.5-flash"

os.environ["GOOGLE_API_KEY"] = config.GOOGLE_API_KEY.strip()

# -----------------------------
# Create LlmAgent (chatbot)
# -----------------------------
root_agent = LlmAgent(
    model=MODEL,
    name="ChatAgent",
    instruction="""
    You are a helpful assistant. Respond politely and concisely to user questions.
    """
)

# -----------------------------
# Setup In-Memory Services
# -----------------------------
session_service = InMemorySessionService()
artifact_service = InMemoryArtifactService()

# Create a new session
await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)

# -----------------------------
# Setup Runner
# -----------------------------
runner = Runner(
    agent=root_agent,
    app_name=APP_NAME,
    session_service=session_service,
    artifact_service=artifact_service
)

# -----------------------------
# Chat function
# -----------------------------
async def chat(user_input):
    content = types.Content(
        role="user",
        parts=[types.Part(text=user_input)]
    )
    
    events = runner.run_async(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=content
    )
    
    async for event in events:
        # Print final response only
        if event.is_final_response() and event.content and event.content.parts:
            text = event.content.parts[0].text
            print("ChatAgent:", text)

# -----------------------------
# Chat
# -----------------------------
await chat("Hello! Can you suggest a fun weekend activity?")
await chat("Give me a short summary of Artificial Intelligence.")


In [ ]:
# Single-Speaker TTS
from google import genai
from google.genai import types
import wave

# Set up the wave file to save the output:
def wave_file(filename, pcm, channels=1, rate=24000, sample_width=2):
   with wave.open(filename, "wb") as wf:
      wf.setnchannels(channels)
      wf.setsampwidth(sample_width)
      wf.setframerate(rate)
      wf.writeframes(pcm)

client = genai.Client()

response = client.models.generate_content(
   model="gemini-2.5-flash-preview-tts",
   contents="Say cheerfully: Have a wonderful day!",
   config=types.GenerateContentConfig(
      response_modalities=["AUDIO"],
      speech_config=types.SpeechConfig(
         voice_config=types.VoiceConfig(
            prebuilt_voice_config=types.PrebuiltVoiceConfig(
               voice_name='Kore',
            )
         )
      ),
   )
)

data = response.candidates[0].content.parts[0].inline_data.data

file_name='out.wav'
wave_file(file_name, data) # Saves the file to current directory

In [1]:
import os
import time
import asyncio
from datetime import datetime
from google import genai
from google.genai import types
from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.tools.google_search_tool import google_search
from google.adk.tools import FunctionTool
from google.adk.sessions import InMemorySessionService
from google.adk.artifacts import InMemoryArtifactService
from google.adk.runners import Runner
from config import config

# ============================================================
# CONFIG
# ============================================================
APP_NAME = "Dynamic-Topic-Speech-Pipeline"
USER_ID = "user_001"
SESSION_ID = "session_001"
TEXT_MODEL = "gemini-2.5-flash"
TTS_MODEL = "gemini-2.5-flash-preview-tts"

os.environ["GOOGLE_API_KEY"] = config.GOOGLE_API_KEY.strip()

# ============================================================
# HELPER: Save PCM data to WAV
# ============================================================
import wave

def save_wave_file(filename, pcm, channels=1, rate=24000, sample_width=2):
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    with wave.open(filename, "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(sample_width)
        wf.setframerate(rate)
        wf.writeframes(pcm)

# ============================================================
# TOOL: TTS GENERATION
# ============================================================
async def generate_tts_tool(speech_text: str, topic_name: str) -> dict:
    """
    Generate TTS audio from the given speech text.
    """
    client = genai.Client()
    
    print("\n🎤 Generating Speech Audio...\n")
    
    response = client.models.generate_content(
        model=TTS_MODEL,
        contents=speech_text,
        config=types.GenerateContentConfig(
            response_modalities=["AUDIO"],
            speech_config=types.SpeechConfig(
                voice_config=types.VoiceConfig(
                    prebuilt_voice_config=types.PrebuiltVoiceConfig(
                        voice_name='Kore'
                    )
                )
            ),
        )
    )
    
    data = response.candidates[0].content.parts[0].inline_data.data
    
    # Clean topic name for filename
    topic_name_clean = "".join(c if c.isalnum() else "_" for c in topic_name.strip())
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = os.path.join("generated_speech", f"{topic_name_clean}_{timestamp}.wav")
    
    save_wave_file(filename, data)
    
    print(f"\n✅ Speech Audio Generated Successfully: {filename}\n")
    
    return {"status": "success", "filename": filename}

generate_tts = FunctionTool(func=generate_tts_tool)

# ============================================================
# AGENT 1 — DEEP RESEARCH AGENT
# ============================================================
research_agent = LlmAgent(
    name="deep_topic_research_agent",
    model=TEXT_MODEL,
    instruction="""
You are an expert researcher tasked with creating a highly detailed, comprehensive report.
Your goal:

1. Perform extensive research on the user-specified topic using Google searches.
2. Gather high-quality information from:
   - Official websites
   - Academic papers / industry reports
   - Press releases
   - News articles
   - Blogs / expert opinions
   - Relevant statistics
3. Aim to produce a detailed document of 3000–4000 words.
4. Ensure all information is accurate, structured, and clear.
5. Include logical flow suitable for turning into a speech.
6. Output the research content ONLY, no summaries, no opinions.

Important: Maintain depth and factual accuracy.
""",
    tools=[google_search]
)

# ============================================================
# AGENT 2 — SPEECH PREPARATION & TTS
# ============================================================
speech_agent = LlmAgent(
    name="speech_preparation_agent",
    model=TEXT_MODEL,
    instruction="""
You are a professional speechwriter.

1. Take the entire research content from the previous agent.
2. Convert it into a coherent, natural, and engaging speech suitable for reading aloud.
3. Output the speech text ONLY — no extra headers or metadata.
4. Pass this speech text and topic name to the TTS tool (generate_tts) to generate audio.
""",
    tools=[generate_tts]
)

# ============================================================
# PIPELINE
# ============================================================
pipeline_agent = SequentialAgent(
    name="dynamic_topic_speech_pipeline",
    sub_agents=[research_agent, speech_agent],
)

# ============================================================
# RUNNER
# ============================================================
async def main(topic_name: str):
    session_service = InMemorySessionService()
    artifact_service = InMemoryArtifactService()
    
    await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=SESSION_ID
    )
    
    runner = Runner(
        agent=pipeline_agent,
        app_name=APP_NAME,
        session_service=session_service,
        artifact_service=artifact_service
    )
    
    user_prompt = f"Create a comprehensive research report on the topic: {topic_name}"
    
    user_message = types.Content(
        role="user",
        parts=[types.Part(text=user_prompt)]
    )
    
    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=user_message
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print("\n🧠 TEXT OUTPUT:\n", part.text)
                elif part.function_call:
                    print("\n🔧 TOOL CALL →", part.function_call.name)
                elif part.function_response:
                    print("\n✅ TOOL RESPONSE →", part.function_response.response)

# ============================================================
# RUN EXAMPLE
# ============================================================
# asyncio.run(main("Artificial Intelligence"))
await main("Blockchain")



🧠 TEXT OUTPUT:
 ## Blockchain: A Comprehensive Research Report

Blockchain technology represents a paradigm shift in how information is stored, shared, and managed across digital networks. Born from the need for a secure, decentralized digital currency, it has evolved into a versatile framework with the potential to revolutionize numerous industries beyond its initial application in cryptocurrencies. At its essence, blockchain is a distributed digital ledger that records transactions across multiple computers in a secure and immutable manner, fostering transparency and trust without the need for a central authority.

### 1. Definition and Core Concepts

A blockchain is fundamentally an advanced database mechanism that allows for transparent information sharing within a business network. It stores data in "blocks" that are linked together in a "chain" using cryptographic principles. Once data is recorded, it becomes chronologically consistent and cannot be deleted or modified without t

D:\Agent-Development-Kit\venv\Lib\site-packages\google\adk\flows\llm_flows\base_llm_flow.py:449: UserWarning: [EXPERIMENTAL] feature FeatureName.PROGRESSIVE_SSE_STREAMING is enabled.
  async for event in agen:



🧠 TEXT OUTPUT:
 Good morning, everyone.

It's truly exciting to stand here today and talk about a technology that is not just changing, but truly *redefining*, our digital world: Blockchain.

Many of you might associate blockchain primarily with cryptocurrencies like Bitcoin, and you wouldn't be wrong. That's where it found its initial footing. But to view blockchain solely through that lens would be like looking at the internet and only seeing email. Blockchain is so much more. It's a fundamental shift in how we store, share, and manage information across digital networks – a paradigm shift that promises transparency, trust, and security without the need for a central authority.

At its heart, a blockchain is an advanced digital ledger. Imagine a series of interconnected digital blocks, each containing a record of transactions, sealed with an unbreakable cryptographic link to the one before it. Once data is added to this chain, it’s practically immutable – it cannot be deleted or mod

In [4]:
import os
import asyncio
from datetime import datetime
from google import genai
from google.genai import types
from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.tools.google_search_tool import google_search
from google.adk.sessions import InMemorySessionService
from google.adk.artifacts import InMemoryArtifactService
from google.adk.runners import Runner
from config import config
import wave

# ============================================================
# CONFIG
# ============================================================
APP_NAME = "Dynamic-Topic-Speech-Pipeline"
USER_ID = "user_001"
SESSION_ID = "session_001"
TEXT_MODEL = "gemini-2.5-flash"
TTS_MODEL = "gemini-2.5-flash-preview-tts"

os.environ["GOOGLE_API_KEY"] = config.GOOGLE_API_KEY.strip()

# ============================================================
# HELPER: Save PCM to WAV
# ============================================================
def save_wave_file(filename, pcm, channels=1, rate=24000, sample_width=2):
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    with wave.open(filename, "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(sample_width)
        wf.setframerate(rate)
        wf.writeframes(pcm)

# ============================================================
# TTS FUNCTION (NOW CALLED DIRECTLY FROM MAIN)
# ============================================================
async def generate_tts_tool(
    speech_text: str,
    topic_name: str,
    voice_name: str,
    language: str
) -> dict:

    client = genai.Client()

    print(f"\n🎤 Generating Speech Audio in '{language}' with voice '{voice_name}'...\n")

    response = client.models.generate_content(
        model=TTS_MODEL,
        contents=speech_text,
        config=types.GenerateContentConfig(
            response_modalities=["AUDIO"],
            speech_config=types.SpeechConfig(
                voice_config=types.VoiceConfig(
                    prebuilt_voice_config=types.PrebuiltVoiceConfig(
                        voice_name=voice_name
                    )
                ),
                language_code=language
            ),
        )
    )

    data = response.candidates[0].content.parts[0].inline_data.data

    topic_name_clean = "".join(c if c.isalnum() else "_" for c in topic_name.strip())
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = os.path.join("generated_speech", f"{topic_name_clean}_{timestamp}.wav")

    save_wave_file(filename, data)

    print(f"\n✅ Speech Audio Generated Successfully: {filename}\n")

    return {"status": "success", "filename": filename}


# ============================================================
# AGENT 1 — RESEARCH
# ============================================================
research_agent = LlmAgent(
    name="deep_topic_research_agent",
    model=TEXT_MODEL,
    instruction="""
You are an expert researcher.

Create a detailed, structured, 3000–4000 word research report
on the given topic using Google search.

Output ONLY the research content.
""",
    tools=[google_search]
)

# ============================================================
# AGENT 2 — SPEECH WRITER (NO TTS HERE)
# ============================================================
speech_agent = LlmAgent(
    name="speech_preparation_agent",
    model=TEXT_MODEL,
    instruction="""
You are a professional speechwriter.

Convert the research document into a natural,
engaging speech suitable for speaking aloud.

Output ONLY the speech text.
"""
)

# ============================================================
# PIPELINE
# ============================================================
pipeline_agent = SequentialAgent(
    name="dynamic_topic_speech_pipeline",
    sub_agents=[research_agent, speech_agent],
)

# ============================================================
# RUNNER
# ============================================================
async def main(topic_name: str, voice_name: str = "Achernar", language: str = "en"):

    session_service = InMemorySessionService()
    artifact_service = InMemoryArtifactService()

    await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=SESSION_ID
    )

    runner = Runner(
        agent=pipeline_agent,
        app_name=APP_NAME,
        session_service=session_service,
        artifact_service=artifact_service
    )

    user_prompt = f"Create a comprehensive research report on the topic: {topic_name}"

    user_message = types.Content(
        role="user",
        parts=[types.Part(text=user_prompt)]
    )

    final_speech_text = ""

    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=user_message
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    final_speech_text += part.text

    # ============================================================
    # CALL TTS DIRECTLY (DETERMINISTIC CONTROL)
    # ============================================================

    await generate_tts_tool(
        speech_text=final_speech_text,
        topic_name=topic_name,
        voice_name=voice_name,
        language=language
    )


# ============================================================
# RUN
# ============================================================

# asyncio.run(main("Artificial Intelligence", voice_name="Zephyr", language="en"))
await main("Impact of AI", voice_name="Zephyr", language="en")


🎤 Generating Speech Audio in 'en' with voice 'Zephyr'...


✅ Speech Audio Generated Successfully: generated_speech\Impact_of_AI_20260225_151905.wav



In [1]:
import os
import asyncio
from datetime import datetime
from google import genai
from google.genai import types
from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.tools.google_search_tool import google_search
from google.adk.tools import FunctionTool
from google.adk.sessions import InMemorySessionService
from google.adk.artifacts import InMemoryArtifactService
from google.adk.runners import Runner
import wave
from config import config

# ============================================================
# CONFIG
# ============================================================
APP_NAME = "Dynamic-Topic-Speech-Pipeline"
USER_ID = "user_001"
SESSION_ID = "session_001"
TEXT_MODEL = "gemini-2.5-flash"
TTS_MODEL = "gemini-2.5-flash-preview-tts"

os.environ["GOOGLE_API_KEY"] = config.GOOGLE_API_KEY.strip()

# ============================================================
# HELPER: Save PCM to WAV
# ============================================================
def save_wave_file(filename, pcm, channels=1, rate=24000, sample_width=2):
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    with wave.open(filename, "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(sample_width)
        wf.setframerate(rate)
        wf.writeframes(pcm)

# ============================================================
# TOOL: TTS GENERATION (NO DEFAULTS — REQUIRED PARAMS)
# ============================================================
async def generate_tts_tool(
    speech_text: str,
    topic_name: str,
    voice_name: str,
    language: str
) -> dict:

    client = genai.Client()

    print(f"\n🎤 Generating Speech Audio in '{language}' with voice '{voice_name}'...\n")

    response = client.models.generate_content(
        model=TTS_MODEL,
        contents=speech_text,
        config=types.GenerateContentConfig(
            response_modalities=["AUDIO"],
            speech_config=types.SpeechConfig(
                voice_config=types.VoiceConfig(
                    prebuilt_voice_config=types.PrebuiltVoiceConfig(
                        voice_name=voice_name
                    )
                ),
                language_code=language
            ),
        )
    )

    data = response.candidates[0].content.parts[0].inline_data.data

    topic_name_clean = "".join(c if c.isalnum() else "_" for c in topic_name.strip())
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = os.path.join("generated_speech", f"{topic_name_clean}_{timestamp}.wav")

    save_wave_file(filename, data)

    print(f"\n✅ Speech Audio Generated Successfully: {filename}\n")

    return {"status": "success", "filename": filename}


generate_tts = FunctionTool(func=generate_tts_tool)

# ============================================================
# AGENT 1 — RESEARCH
# ============================================================
research_agent = LlmAgent(
    name="deep_topic_research_agent",
    model=TEXT_MODEL,
    instruction="""
You are an expert researcher.

1. Perform extensive research using reliable sources.
2. Gather information from websites, news, reports, blogs, and statistics.
3. Produce 3000–4000 words.
4. Ensure factual accuracy.
5. Output ONLY the research content.
""",
    tools=[google_search]
)

# ============================================================
# AGENT 2 — SPEECH WRITER
# ============================================================
speech_agent = LlmAgent(
    name="speech_preparation_agent",
    model=TEXT_MODEL,
    instruction="""
You are a professional speechwriter.

Convert the research into a natural, engaging speech.
Output ONLY the speech text.
"""
)

# ============================================================
# AGENT 3 — TRANSLATION
# ============================================================
translation_agent = LlmAgent(
    name="translation_agent",
    model=TEXT_MODEL,
    instruction="""
You are a professional translator.

Translate the speech into the target language specified
in the user request.

Preserve natural spoken style.
Do not summarize or shorten.
Output ONLY the translated text.
"""
)

# ============================================================
# AGENT 4 — TTS CONTROLLER
# ============================================================
tts_agent = LlmAgent(
    name="tts_agent",
    model=TEXT_MODEL,
    instruction="""
You are a TTS controller.

You MUST call generate_tts with ALL required arguments:

- speech_text (translated speech)
- topic_name
- voice_name (exactly as specified in the user request)
- language (exactly as specified in the user request)

Do not change or invent parameter values.
""",
    tools=[generate_tts]
)

# ============================================================
# PIPELINE
# ============================================================
pipeline_agent = SequentialAgent(
    name="dynamic_topic_speech_pipeline",
    sub_agents=[
        research_agent,
        speech_agent,
        translation_agent,
        tts_agent
    ]
)

# ============================================================
# RUNNER
# ============================================================
async def main(topic_name: str, voice_name: str, language: str):

    session_service = InMemorySessionService()
    artifact_service = InMemoryArtifactService()

    await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=SESSION_ID
    )

    runner = Runner(
        agent=pipeline_agent,
        app_name=APP_NAME,
        session_service=session_service,
        artifact_service=artifact_service
    )

    # Inject runtime config into prompt
    user_prompt = f"""
Create a comprehensive research report on the topic: {topic_name}

Target translation language: {language}
TTS voice to use: {voice_name}

Instructions:
- Translate final speech into {language}
- Call generate_tts with voice_name="{voice_name}"
- Call generate_tts with language="{language}"
"""

    user_message = types.Content(
        role="user",
        parts=[types.Part(text=user_prompt)]
    )

    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=user_message
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print("\n🧠 TEXT OUTPUT:\n", part.text)
                elif part.function_call:
                    print("\n🔧 TOOL CALL →", part.function_call.name)
                elif part.function_response:
                    print("\n✅ TOOL RESPONSE →", part.function_response.response)


# ============================================================
# EXAMPLE RUN
# ============================================================

# asyncio.run(main("Indian Culture", voice_name="Achird", language="hi"))
await main("Indian Culture", voice_name="Achird", language="hi")



🧠 TEXT OUTPUT:
 भारत की संस्कृति विश्व की सबसे पुरानी और विविध संस्कृतियों में से एक है, जिसकी जड़ें लगभग 4,500 साल पुरानी सिंधु घाटी सभ्यता में मिलती हैं. इसे अक्सर "सा प्रथमा संस्कृति विश्ववारा" के रूप में वर्णित किया जाता है - अर्थात दुनिया की पहली और सर्वोच्च संस्कृति. भारतीय संस्कृति सामाजिक मानदंडों, नैतिक मूल्यों, पारंपरिक रीति-रिवाजों, विश्वास प्रणालियों, राजनीतिक प्रणालियों, कलाकृतियों और प्रौद्योगिकियों का एक संग्रह है, जो जातीय-भाषाई रूप से विविध भारतीय उपमहाद्वीप से जुड़ी हैं. यह अपनी "अनेकता में एकता" के लिए विख्यात है, एक अनूठी विशेषता जो इसके विशाल भौगोलिक विस्तार में विभिन्न संस्कृतियों, भाषाओं, धर्मों और परंपराओं के सामंजस्यपूर्ण सह-अस्तित्व को दर्शाती है. इस विशाल विविधता के बावजूद, एक अरब से अधिक लोगों को भारतीय होने की सामूहिक पहचान एक सूत्र में बांधती है. यह एकता साझा इतिहास, मूल्यों और एक सामूहिक पहचान में निहित है जो मतभेदों से परे है.

भारतीय संस्कृति के कई तत्वों, जैसे भारतीय धर्म, गणित, दर्शन, व्यंजन, भाषाएँ, नृत्य, संगीत और सिनेमा ने इंडोस्फीयर, ग्रेटर इंडिय

D:\Agent-Development-Kit\venv\Lib\site-packages\google\adk\flows\llm_flows\base_llm_flow.py:449: UserWarning: [EXPERIMENTAL] feature FeatureName.PROGRESSIVE_SSE_STREAMING is enabled.
  async for event in agen:



🔧 TOOL CALL → generate_tts_tool

🎤 Generating Speech Audio in 'hi' with voice 'Achird'...


✅ Speech Audio Generated Successfully: generated_speech\Indian_Culture_20260225_152838.wav


✅ TOOL RESPONSE → {'status': 'success', 'filename': 'generated_speech\\Indian_Culture_20260225_152838.wav'}

🧠 TEXT OUTPUT:
 The TTS audio has been successfully generated and saved with the filename `generated_speech\\Indian_Culture_20260225_152838.wav`.
